In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
from pathlib import Path
import re
import seaborn as sns

#### Define directories

In [ ]:
ROOT = Path.cwd().resolve()

DATA = (ROOT / "data").resolve()

eda_df = pd.read_parquet(DATA/"eda_dataset.parquet")

#### Read the data in

In [ ]:
eda_df.columns

#### Modeling target decision

In [ ]:
target_decision = pd.DataFrame({
    "Decision": [
        "Outcome for reporting",
        "Modeling target",
        "Transformation method",
        "Justification"
    ],
    "Value": [
        "stdzd_amt_per_service (raw, $/service)",
        "log1p(stdzd_amt_per_service) (stored as log_stdzd_amt_per_service)",
        "TTR with log1p and expm1.",
        "Heavy-tailed raw outcome. Log reduces tail dominance and improves stability."
    ]
})
target_decision

#### Training inclusion filter

In [ ]:
# Define training inclusion mask explicitly (even if eda_df is already filtered)
train_mask = (
    (eda_df["services"] >= 11) &
    (eda_df["stdzd_amt_per_service"].notna()) &
    (eda_df["stdzd_amt_per_service"] >= 0)
)

train_filter_summary = pd.DataFrame({
    "Metric": [
        "Rows in eda_df",
        "Rows meeting training filter",
        "Share kept",
        "Unique NPIs (all)",
        "Unique NPIs (kept)",
        "Years present (all)",
        "Years present (kept)",
    ],
    "Value": [
        len(eda_df),
        int(train_mask.sum()),
        float(train_mask.mean()),
        eda_df["Rndrng_NPI"].nunique(),
        eda_df.loc[train_mask, "Rndrng_NPI"].nunique(),
        sorted(eda_df["Year"].unique().tolist()),
        sorted(eda_df.loc[train_mask, "Year"].unique().tolist()),
    ]
})

train_filter_summary

#### Define feature list:

In [ ]:
# Categorical features (to encode)
cat_features = [
    "rbcs_family_desc",    # or use "RBCS_FamNumb" instead, but pick one consistently
    "Place_Of_Srvc",
    "provider_type",
    "state",
    "ruca_bucket",
]

# Numeric features
num_features = [
    "bene_avg_risk_score",
    "years_since_enumeration",
    "log_services",
    "log_benes",
    "p_cancer6", "p_diabetes", "p_ckd", "p_copd", "p_htn",
]

# Final target
target_col = "stdzd_amt_per_service"

# Exclusions (documented)
excluded = [
    # raw cost outcomes besides target
    "log_stdzd_amt_per_service", "allowed_amt_per_service", "payment_amt_per_service", "submitted_charge_per_service",
    # totals derived from outcomes (spend columns)
    "stdzd_spend", "allowed_spend", "payment_spend", "submitted_spend",
    # flags/buckets used for slicing, not for training features
    "is_top_1pct_stdzd_amt_per_service", "svc_bucket", "services_bins", "services_custom", "services_custom2",
    # provider-year totals (often avoided to prevent scale leakage; can revisit intentionally later)
    "tot_mdcr_stdzd_amt",
]

features_table = pd.DataFrame({
    "Type": (["categorical"] * len(cat_features)) + (["numeric"] * len(num_features)) + (["target"] * 1),
    "Column": cat_features + num_features + [target_col]
})

features_table

This is our intended modeling feature set.

Categorical features
- `rbcs_family_desc`
- `Place_Of_Srvc`
- `provider_type`
- `state`
- `ruca_bucket`

Interpretation:
- These explain systematic price differences due to:
    - what service is being delivered,
    - where it is delivered,
    - what specialty is delivering it,
    - geography and rurality.

Numeric features
- risk and experience:
    - `bene_avg_risk_score`
    - `years_since_enumeration`
- exposure/intensity controls (log):
    - `log_services`
    - `log_benes`
- case mix proportions:
    - `p_cancer6`, `p_diabetes`, `p_ckd`, `p_copd`, `p_htn`

Interpretation:
- This is a classic “risk adjustment + context” set.
- We are not leaking the target because you excluded spend totals and other cost measures.

Modeling implication
- We picked `rbcs_family_desc` over `RBCS_FamNumb`. Although ID is often cleaner, desc is often fine too.

#### Quick availability check:

In [ ]:
missing_cols = [c for c in (cat_features + num_features + [target_col]) if c not in eda_df.columns]
missing_cols

`missing_cols` gives empty list `[]`

Interpretation
- Everything we plan to model exists in our dataframe.
- This prevents “modeling notebook surprises.”

#### Create split masks

In [ ]:
train_years = [2021, 2022]
test_years = [2023]

mask_train = eda_df["Year"].isin(train_years)
mask_test = eda_df["Year"].isin(test_years)

split_counts = pd.DataFrame({
    "Split": ["train", "test"],
    "Years": [train_years, test_years],
    "Rows": [int(mask_train.sum()), int(mask_test.sum())],
    "Unique NPIs": [eda_df.loc[mask_train, "Rndrng_NPI"].nunique(), eda_df.loc[mask_test, "Rndrng_NPI"].nunique()],
    "Spend share": [
        float(eda_df.loc[mask_train, "stdzd_spend"].sum() / eda_df["stdzd_spend"].sum()),
        float(eda_df.loc[mask_test, "stdzd_spend"].sum() / eda_df["stdzd_spend"].sum()),
    ],
})
split_counts

The output
- Train:
    - 213,296 rows
    - 19,838 NPIs
    - 66.6% spend
- Test:
    - 105,026 rows
    - 19,226 NPIs
    - 33.4% spend

Interpretation
- YoWeu have a healthy split. About one-third of dollars are in the test year.
- The train and test have similar provider counts, which is good for generalization tests.

Modeling implication
- This is a realistic production-like test: learn patterns from earlier years, apply to the next year.


#### Provider overlap (leakage / generalization check):

In [ ]:
npi_train = set(eda_df.loc[mask_train, "Rndrng_NPI"].unique().tolist())
npi_test = set(eda_df.loc[mask_test, "Rndrng_NPI"].unique().tolist())

overlap = npi_train.intersection(npi_test)
test_only = npi_test - npi_train

provider_overlap_tbl = pd.DataFrame({
    "Metric": ["Train NPIs", "Test NPIs", "Overlap NPIs", "Test-only NPIs"],
    "Value": [len(npi_train), len(npi_test), len(overlap), len(test_only)],
    "Share of test NPIs": [
        np.nan,
        1.0,
        len(overlap)/len(npi_test) if len(npi_test) else np.nan,
        len(test_only)/len(npi_test) if len(npi_test) else np.nan,
    ]
})
provider_overlap_tbl

The `provider_overlap_tbl` output table:
- `Test NPIs`: 19,226
- `Overlap with train`: 18,145 (94.38%)
- `Test-only`: 1,081 (5.62%)

Interpretation
- Most test providers were seen in train. That means your evaluation is mostly:
    - “new year for known providers” (easier)
- But you still have a meaningful cold-start set:
    - 1,081 providers

Modeling implication
- You should report performance separately for:
    - seen providers (in train)
    - unseen providers (test-only)

Because those are different deployment realities.

#### Planned slice keys table:

In [ ]:
eval_plan = pd.DataFrame({
    "Category": [
        "Primary metrics",
        "Target scale",
        "Core slices (report)",
        "Stability slices (flagging)",
        "Tail diagnostic"
    ],
    "Plan": [
        "MAE, RMSE",
        "log of stdzd_amt_per_service via TTR (log1p)",
        "provider_type, Place_Of_Srvc, ruca_bucket, state",
        "svc_bucket and services>=50 / >=100 subsets",
        "Compare tail vs non-tail behavior without changing labels"
    ]
})
eval_plan

#### Modeling readiness summary table (final contract):

In [ ]:
readiness_contract = pd.DataFrame({
    "Decision Area": [
        "Dataset grain",
        "Final modeling target",
        "Training inclusion",
        "Tail handling",
        "Visualization-only clipping",
        "Post-model flagging threshold",
        "Features (categorical)",
        "Features (numeric)",
        "Split plan",
        "Evaluation slices"
    ],
    "Final Choice": [
        "provider-year-RBCS family-place of service",
        "log of stdzd_amt_per_service via TTR = log1p(stdzd_amt_per_service)",
        "services >= 11; stdzd_amt_per_service not null and >= 0",
        "Keep tail; do not winsorize labels; use tail flag for diagnostics",
        "Allow p99 clipping for readability in plots only",
        "Flagging candidates evaluated on services >= 50 (and >= 100 sensitivity)",
        ", ".join(cat_features),
        ", ".join(num_features),
        "Train 2021–2022, Test 2023",
        "provider_type, Place_Of_Srvc, ruca_bucket, svc_bucket, tail flag (diagnostic)"
    ],
    "Why defensible": [
        "Matches EDA grain and planned use case",
        "Controls heavy tail while preserving signal",
        "Reliability threshold reduces denominator noise",
        "Tail appears real and interpretable (not pure error). Log target handles skew",
        "Prevents misleading plots without altering training distribution",
        "Reduces false positives from low-volume instability",
        "Captures key context (service, POS, specialty, geography, rurality)",
        "Captures case-mix + intensity + experience + comorbidity composition",
        "Temporal generalization is the real-world test",
        "Ensures interpretability and stability across key segments"
    ]
})

readiness_contract

#### The data frame for modeling

In [ ]:
model_cols = cat_features + num_features + [target_col, "Year", "Rndrng_NPI", "services", "log_stdzd_amt_per_service"]

model_df = eda_df.loc[train_mask, model_cols].copy()

train_df = model_df[model_df["Year"].isin(train_years)].copy()
test_df = model_df[model_df["Year"].isin(test_years)].copy()

train_df.shape, test_df.shape

#### Missingness in features used in modeling

In [ ]:
feature_missing = (
    model_df[cat_features + num_features + [target_col]]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("pct_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)
feature_missing

The `feature_missing` is the one we should pay attention to before training.

We have missingness in:
- `p_copd`: 4.59%
- `p_ckd`: 2.05%
- `p_diabetes`: 1.15%
- `p_cancer6`: 0.86%
- `years_since_enumeration`: 0.45%
- `p_htn`: 0.05%
Everything else: 0%

Interpret each variable’s missingness and what it implies

- `p_copd` (4.59% missing)
    - This is the highest missingness among our features.
    - Likely causes:
        - certain provider-year records lack the COPD percentage due to suppression, reporting rules, or merge gaps.
    - Modeling risk:
        - dropping rows would throw away ~4.6% of our data for one feature.
    - Recommended handling:
        - impute missing with a neutral value (commonly 0) plus a missingness indicator, or
        - impute with median and add indicator.
    - Why indicator matters:
        - missingness might correlate with provider type or geography, so the “missing” itself can carry signal.

- `p_ckd` (2.05% missing)
    - Similar logic, lower magnitude.
    - We should handle it the same way as p_copd for consistency.

- `p_diabetes` (1.15% missing)
    - Small but non-trivial.
    - Same treatment.

- `p_cancer6` (0.86% missing)
    - Small. Same treatment.
    - This one is especially sensitive conceptually in oncology, so do not silently drop rows.

- `years_since_enumeration` (0.45% missing)
    - This is “provider experience proxy.”
    - Missingness likely means:
        - NPI enumeration date missing upstream, or mapping failed.
    - Modeling risk:
        - leaving it missing can break some models.
    - Handling:
        - impute median and add missingness flag is the safest.

- `p_htn` (0.05% missing)
    - Very small. Still handle systematically (same imputation pattern).
    - Consistency matters. We do not want special-case logic for one feature.

- All categoricals have 0% missing
    - That is excellent. It means our slicing features are complete.
    - Especially important for:
        - `provider_type`, `Place_Of_Srvc`, `ruca_bucket`.

- `bene_avg_risk_score` has 0% missing
    - This is great because it is typically our primary adjustment feature.

Modeling implication
- We need a missingness strategy as part of Notebook 10 or the first modeling notebook.
- The best practice approach here is:

1.	For each numeric feature with missing:

- create `is_missing_<feature>` indicator

2.	Impute missing values:

- either 0 (for percentage fields) or median

3.	Keep the indicator in the model

This preserves rows, avoids bias from dropping, and allows missingness patterns to be learned.


#### Build X/y

In [ ]:
# Target is log target (your modeling target)
y_train = train_df[target_col].copy()
y_test  = test_df[target_col].copy()

# Keep these for analysis, but not as model predictors
id_cols = ["Rndrng_NPI", "Year"]

# Also exclude raw cost outcome (you do not want leakage)
exclude_from_X = [target_col, "log_stdzd_amt_per_service"] + id_cols

X_train = train_df.drop(columns=exclude_from_X).copy()
X_test  = test_df.drop(columns=exclude_from_X).copy()

Now X_train contains exactly `cat_features + num_features`. 

#### Preprocess with `ColumnTransformer`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None"))
])

cat_pipe_ohe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

preprocess_ohe = ColumnTransformer(
    transformers=[
        ("cat_ohe", cat_pipe_ohe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

#### Fit-transform train, transform test

In [ ]:
X_train_proc      = preprocess.fit_transform(X_train)
X_test_proc       = preprocess.transform(X_test)

X_train_proc_ohe  = preprocess_ohe.fit_transform(X_train)
X_test_proc_ohe   = preprocess_ohe.transform(X_test)

feat_names_ohe = preprocess_ohe.get_feature_names_out()
feat_names_ohe[:10]

In [ ]:
print(X_train_proc.shape)
print(X_test_proc.shape)
print(X_train_proc_ohe.shape)
print(X_test_proc_ohe.shape)

Note: We used `get_feature_names_out()` for debugging and SHAP later.

#### Inspect the reference categories for each categorical variable, i.e., `cat_features`:

Here, we need to 

1. reach into the `preprocess_ohe`, which is the our `ColumnTransformer` and 
2. grab the `cat_ohe` step, the `cat_pipe_ohe`, which is our `Pipeline`, 
3. then we reach into the ``cat_pipe_ohe` `Pipeline`, and 
4. grab the `ohe` step, which is our `OneHotEncoder(...)`

In [ ]:
# A list to store our reference categories
refs = []

# Get the fitted OHE object of the cat_pipe_ohe Pipeline
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]

# categories_ is a list alighed to cat_features
# each entry is an array of the learned categories for that feature 
for col, cats in zip(cat_features, ohe.categories_):
    refs.append({
        "feature": col,
        "reference_dropped":cats[0], # dropped because drop="first"
        "all_categories": list(cats), #optional, can be long for rbcs_family_desc
        "n_categories": len(cats)
    })

ref_table = pd.DataFrame(refs)[["feature","reference_dropped", "n_categories"]]
ref_table

Now we have:
- `X_train_proc`: imputed features
- `X_test_proc`: same columns, same encoder mapping, no leakage

- `X_train_proc_ohe`: one-hot encoded and imputed features
- `X_test_proc_ohe`: same columns, same encoder mapping, no leakage

#### Explain `UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros warnings.warn(msg, UserWarning)`

- During `fit_transform(X_train)`, the encoder learns the set of categories seen in **train** for each categorical feature.
- During `transform(X_test)`, it encountered at least one category in **column [0]** (that is your first categorical feature, `rbcs_family_desc`) that was **not present in train**.
- Because we set `handle_unknown="ignore"`, sklearn does not crash. It encodes those unseen categories as **all zeros across the OHE columns for that feature**.

So the model effectively treats “unseen category” as “none of the known categories”.

Let's add a small check so we know how often it happens (this tells us what fraction of test rows have unseen `rbcs_family_desc`):

In [ ]:
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]
cats = ohe.categories_[0]  # categories for first categorical feature
n_unknown = (~X_test[cat_features[0]].astype(str).isin(cats)).sum()
share_unknown = n_unknown / len(X_test)
n_unknown, share_unknown

The result basically says:

- **Only 2 rows in your entire 2023 test set** have an `rbcs_family_desc` value that never appeared in 2021–2022 training.
- That is **0.0019% of test rows** (about 1 in 52,500 rows).

So the warning is totally benign here.

In [ ]:
from preprocessing import CorrelationThreshold

# ==========================================
# RE-SYNC & PLOT (RUN THIS WHOLE BLOCK)
# ==========================================

# 0. Build a dense DataFrame with correct column names, then compute correlation
import scipy.sparse as sp

# X_train_proc_ohe is csr_matrix, feat_names_ohe length = 193
X_train_proc_ohe_dense = X_train_proc_ohe.toarray() if sp.issparse(X_train_proc_ohe) else np.asarray(X_train_proc_ohe)
X_train_proc_ohe_df = pd.DataFrame(X_train_proc_ohe_dense, columns=feat_names_ohe)

# 1. RE-CALCULATE DROPS (Ensure the list is fresh!)
corr_selector = CorrelationThreshold(threshold=0.9)
corr_selector.fit(X_train_proc_ohe_df)
dropped_cols = corr_selector.to_drop_

# 2. RE-CALCULATE SUBSET DATA
# We need to rebuild the subset_corr to match the fresh dropped_cols list
if len(dropped_cols) > 0:
    # Find partners again
    corr_matrix = X_train_proc_ohe_df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    partners = []
    for dropped in dropped_cols:
        # Find the feature kept
        partner_match = upper.index[upper[dropped] > 0.9].tolist()
        if partner_match:
            partners.append(partner_match[0])
            
    # Combine lists
    features_to_plot = list(set(dropped_cols + partners))
    subset_corr = X_train_proc_ohe_df[features_to_plot].corr()

    # 3. PLOT
    plt.figure(figsize=(16, 14))
    ax = plt.gca()

    # Heatmap
    mask = np.triu(np.ones_like(subset_corr, dtype=bool))
    sns.heatmap(
        subset_corr,
        mask=mask,
        cmap='coolwarm',
        center=0,
        square=True,
        linewidths=.5,
        cbar_kws={"shrink": .5},
        annot=False, 
        ax=ax
    )

    # 4. HIGHLIGHTING LOOP
    # Fix X-axis labels
    new_x_labels = []
    for label in ax.get_xticklabels():
        text = label.get_text()
        if text in dropped_cols:
            label.set_color('red')
            label.set_weight('bold')
            label.set_text(f"[DROP] {text}") 
        new_x_labels.append(label)
    ax.set_xticklabels(new_x_labels)

    # Fix Y-axis labels
    new_y_labels = []
    for label in ax.get_yticklabels():
        text = label.get_text()
        if text in dropped_cols:
            label.set_color('red')
            label.set_weight('bold')
            label.set_text(f"[DROP] {text}")
        new_y_labels.append(label)
    ax.set_yticklabels(new_y_labels)

    plt.title(f"Redundancy Audit: Features marked with [DROP] will be removed ({len(dropped_cols)} total)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

else:
    print("Zero redundancies found. Nothing to plot!")

In [ ]:
X_train_proc_ohe.shape, len(feat_names_ohe), [c for c in feat_names_ohe if "ruca_bucket" in c]

In [ ]:
# What categories are actually present in train for ruca_bucket?
X_train["ruca_bucket"].astype(str).value_counts(dropna=False).head(10)

In [ ]:
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]
# index of ruca_bucket within cat_features
ruca_idx = cat_features.index("ruca_bucket")
ohe.categories_[ruca_idx]

In [ ]:
dropped_cols

#### Extract the feature names for the `X_train_proc`

Remember that `X_train_proc` is a matrix, so it does not have "column names". 

We need to extract the column names from the `Pipeline` called `cat_pipe`:

In [ ]:
feat_names = preprocess.get_feature_names_out()
feat_names[:10]

In [ ]:
len(feat_names)

#### Create a pandas dataframe from the `X_train_proc` which is a sparse `csr_matrix` 

First, we need to turn it into a dense matrix

Second, we turn the dense matrix into a Pandas DataFrame

In [ ]:
# X_train_proc_ohe is csr_matrix, feat_names length = 193
X_train_proc_dense = X_train_proc.toarray() if sp.issparse(X_train_proc) else np.asarray(X_train_proc)
X_train_proc_df = pd.DataFrame(X_train_proc_dense, columns=feat_names)

In [ ]:
X_train_proc_df.head()

> Now we can inspect the `X_train_proc_df` just like the `X_train_proc_ohe_df`, if desired. We could apply the same logic to the `X_test_proc` and `X_test_proc_ohe` also. 

### Audit correlations on the numeric block only

#### Extract the transformed numeric matrix (with missingness indicators)

In [ ]:
num_features

In [ ]:
# Grab the fitted numeric pipeline (the num_pipe)
num_pipe_fitted = preprocess_ohe.named_transformers_["num"]

# Transform only the numeric columns 
X_train_num_proc = num_pipe_fitted.transform(X_train[num_features]) # numpy array, small width

#### Get the numeric feature names (including indicators)

In [ ]:
num_feat_names = num_pipe_fitted.get_feature_names_out(num_features)
num_feat_names

#### Run your CorrelationThreshold on the numeric DataFrame

In [ ]:
X_train_num_df = pd.DataFrame(X_train_num_proc, columns=num_feat_names, index=X_train.index)

corr_selector_num = CorrelationThreshold(threshold=0.9)
corr_selector_num.fit(X_train_num_df)

dropped_num_cols = corr_selector_num.to_drop_
dropped_num_cols

### Feature Extractor

Here I define a feature extraction function that systematically checks a model's attributes to identify the object, peel off any wrappers, grab the actual model, extract the coefficients or feature importances, construct a dataframe ready to for the next function to plot. 

In [ ]:
import numpy as np
import pandas as pd

def extract_model_features(model_object, feat_names):
    """
    Extract feature names and weights (coefficients or importances).

    Parameters
    ----------
    model_object : fitted estimator
        Can be Pipeline, GridSearchCV, TransformedTargetRegressor, or a plain estimator.
    feat_names : array-like
        Feature names that align with the model's final input space (post-preprocessing).
        Example: preprocess_ohe.get_feature_names_out()

    Returns
    -------
    pd.DataFrame with columns:
      Feature, Coefficient/Importance, Abs_Weight
    """

    # 1) unwrap GridSearchCV
    obj = model_object
    if hasattr(obj, "best_estimator_"):
        obj = obj.best_estimator_

    # 2) unwrap TransformedTargetRegressor
    if hasattr(obj, "regressor_"):
        obj = obj.regressor_

    # 3) identify final fitted estimator
    # If Pipeline, final step is last named step
    if hasattr(obj, "named_steps"):
        final_step_name = list(obj.named_steps.keys())[-1]
        final_model = obj.named_steps[final_step_name]
    else:
        final_model = obj

    current_features = np.array(feat_names)

    # 4) extract weights
    metric_name = None
    weights = None

    if hasattr(final_model, "coef_"):
        weights = final_model.coef_
        metric_name = "Coefficient"

        # Handle shape (1, n_features) or (n_targets, n_features)
        weights = np.asarray(weights)
        if weights.ndim == 2:
            # common single-target 2D shapes
            if 1 in weights.shape:
                weights = weights.ravel()
            else:
                raise ValueError(
                    f"coef_ is 2D with shape {weights.shape}. "
                    "This looks like multioutput. Decide which target to plot."
                )

        # Optional: drop exact/near zeros (useful for Lasso/ElasticNet)
        mask = np.abs(weights) > 1e-5
        current_features = current_features[mask]
        weights = weights[mask]

    elif hasattr(final_model, "feature_importances_"):
        weights = np.asarray(final_model.feature_importances_)
        metric_name = "Importance"

    elif hasattr(final_model, "get_feature_importance"):
        weights = np.asarray(final_model.get_feature_importance())
        metric_name = "Importance"

    elif hasattr(final_model, "estimators_"):
        print("VotingRegressor has no single weight vector. Plot base estimators individually.")
        return pd.DataFrame()

    else:
        raise TypeError(f"Unsupported model type for extraction: {type(final_model)}")

    # 5) sanity check alignment
    if len(current_features) != len(weights):
        raise ValueError(
            f"Feature name mismatch. len(features)={len(current_features)} "
            f"but len(weights)={len(weights)}. "
            "Make sure feat_names matches the model's final input space."
        )

    df = pd.DataFrame({
        "Feature": current_features,
        metric_name: weights
    })
    df["Abs_Weight"] = df[metric_name].abs()
    return df.sort_values("Abs_Weight", ascending=False)

### Feature Impact Visuzlization Function

Here I defined a plotting function that takes in the dataframe containing a model's coefficients or feature importance, sorts the top 20 features by the absolute values of their coefficients or importances (i.e., weights), then plots them as horizontal bar plot.

In [ ]:
# ==========================================
# FEATURE IMPACT VISUALIZATION FUNCTION
# ==========================================

def plot_feature_impact(df, title="Feature Impact", top_n=20):
    if df is None or df.empty:
        print("No data to plot.")
        return

    # Pick metric column
    if "Coefficient" in df.columns:
        metric_col = "Coefficient"
        is_coef = True
    elif "Importance" in df.columns:
        metric_col = "Importance"
        is_coef = False
    else:
        raise ValueError("df must contain either 'Coefficient' or 'Importance' column.")

    # Ensure sorted by absolute weight, then take top_n
    if "Abs_Weight" not in df.columns:
        df = df.copy()
        df["Abs_Weight"] = df[metric_col].abs()

    plot_df = (
        df.sort_values("Abs_Weight", ascending=False)
          .head(top_n)
          .sort_values("Abs_Weight", ascending=True)  # for horizontal bar readability
    )

    plt.figure(figsize=(10, 8))

    if is_coef:
        colors = ["green" if x > 0 else "red" for x in plot_df[metric_col]]
        xlabel = "Impact on log1p(stdzd_amt_per_service) (TTR space)"
    else:
        colors = "skyblue"
        xlabel = "Feature importance"

    plt.barh(plot_df["Feature"], plot_df[metric_col], color=colors)

    if is_coef:
        plt.axvline(x=0, color="black", linestyle="--", linewidth=0.8)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Features")
    plt.tight_layout()
    plt.show()

## Start modeling

### OLS

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

- Define the inner OLS pipe

In [ ]:
ols_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("scaler", StandardScaler(with_mean=False)),  # sparse safe
    ("model", LinearRegression())
])

- wrap the inner pipe inside a `TransformedTargetRegressor()` (TTR)

In [ ]:
ols_full_model = TransformedTargetRegressor(
    regressor=ols_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

- Fit the model

In [ ]:
ols_full_model.fit(X_train, y_train)

- Diagnostics

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

pred_test = ols_full_model.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

print("OLS (Raw y + TTR(log1p))")
print("Train R2:", ols_full_model.score(X_train, y_train))
print("Test  R2:", ols_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)

1. It's likely he relationship is not well captured by a single global linear surface in this feature space (even after the log transform)
2. The RMSE being much larger than the MAE is a classic sign that a small fraction of predictions are very wrong (heavy tail, hard categories, or rare combinations). That is consistent with our EDA.

#### Investigate the reason for poor performance

- Let's make sure we did not accidentally score on the wrong target scale.

In [ ]:
from sklearn.metrics import r2_score

pred_test = ols_full_model.predict(X_test)

r2_dollars = r2_score(y_test, pred_test)

r2_log = r2_score(np.log1p(y_test), np.log1p(pred_test.clip(min=0)))

r2_dollars, r2_log

A) The OLS model is much better at ranking and relative cost than at matching dollars

An **R² of ~0.77 in log space** says the linear model is capturing a lot of the systematic structure in **log1p(cost)**.

But **R² of ~0.22 in dollars** says that once we convert back to dollars, the remaining errors (especially for high-cost cases) explode in magnitude and dominate the variance.

This is exactly what heavy-tailed outcomes do: a small number of expensive rows contribute a huge fraction of dollar variance.

B) The log transform changes the loss geometry

In log space, being off by (say) 30% and being off by 2x are “closer” than they look in dollars.

In dollars, those same misses can be hundreds or thousands of dollars and they dominate SSE, so dollar-scale R² drops.

C) The MAE and RMSE already hinted at this

MAE ~$32 but RMSE ~$152 means “most points are okay, but a few are very wrong in dollars”. Those few are usually the tail.

In [ ]:
pred_test = ols_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary

**Non-tail (False)**

- n = 104,107 test rows
- MAE ≈ $27.93
- RMSE ≈ $50.95
- Mean actual y (y_mean) ≈ $87.04
- Mean prediction (pred_mean) ≈ $85.14

Interpretation:

- On the bulk of the data, the model is roughly centered correctly (mean prediction close to mean actual).
- Error levels are moderate relative to the mean, and consistent with our earlier “R2 in dollars is low but R2 in log space is high” observation.

**Tail (True, top 1% cost per service)**

- n = 919 test rows
- MAE ≈ $514.68
- RMSE ≈ $1532.28
- Mean actual y (y_mean) ≈ $885.80
- Mean prediction (pred_mean) ≈ $372.09

Interpretation:

- The model **massively underpredicts** the tail on average.
- The mean prediction is **about 42%** of the mean actual.

A quick calculation we can do mentally:

- Bias in tail mean ≈ $885.8 − $372.1 ≈ **$513.7**, which is basically the MAE.
- That’s a strong sign the dominant error mode in the tail is “systematic underprediction,” not just noisy scatter.

#### Two small follow-up diagnostics that will help us confirm the story

**1. Tail mean ratio and bias**

In [ ]:
tail = tail_summary.loc[True]
ratio = tail["pred_mean"] / tail["y_mean"]
bias = tail["pred_mean"] - tail["y_mean"]
ratio, bias

- **Ratio = 0.4201**
    - On average, in the tail the OLS model predicts only **42%** of the true cost per service.
- **Bias = −$513.71**
    - On average, it’s under by about **$514** per row in the tail.

This aligns almost exactly with the tail MAE we saw (**~$514.68**). That’s not a coincidence. It means the dominant error mode in the tail is a consistent downward bias, not random noise.

**2. Tail share of total squared error (how much tail dominates RMSE)**

In [ ]:
err_share = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service")["sq_err"]
    .sum()
    .pipe(lambda s: s / s.sum())
)
err_share

We found:

- **Tail rows (True) account for 88.87% of total squared error**
- **Non-tail rows (False) account for 11.13%**

This is the key takeaway:

Even though the tail is a tiny slice of rows (919 out of 105,026, under 1%), it contributes almost **9 out of every 10 “RMSE dollars”** because squared error explodes when we miss large values.

That means:

- Our **overall RMSE in dollars is basically a tail metric**.
- Improvements that help the bulk (non-tail) may barely move RMSE if the tail remains underpredicted.
- A model can look “fine” on non-tail MAE and still look “terrible” overall due to tail.

**3. Quick sanity check (very informative): This will tell us whether the tail errors are mostly “underpredict” vs “overpredict”:**

In [ ]:
tail_rows = test_eval["is_top_1pct_stdzd_amt_per_service"]
signed_err = (test_eval.loc[tail_rows, "pred"] - test_eval.loc[tail_rows, target_col])

signed_err.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])

The tail errors are overwhelmingly **systematic underprediction**, with a few extreme misses that dominate RMSE.

**What each line tells us**

A) Median and percentiles confirm “almost always under”

- **50% (median) = −308.69**
- **75% = −145.17**
- **90% = −76.42**
- Even at the 90th percentile, the error is still negative. That means **at least 90% of tail rows are underpredicted**.

A quick inference we can safely state: **Underprediction is the norm, not an occasional issue.**

B) Only a tiny fraction overpredict

- **max = +125.48**
    
    So the worst overprediction in the tail is only +$125, while the worst underprediction is enormous (see below). That asymmetry is telling.

C) The mean matches the bias

- **mean = −513.71**, exactly what we computed before.
    
    So the “tail bias” number is not a fluke. It’s literally the average signed error.

D) RMSE is being crushed by a few catastrophic misses

- **min = −41,766.47**
- **std = 1,444.38**

That one line explains why tail RMSE is so huge. Squared error makes a single −$41k miss count like thousands of “normal” misses.

**4. What share of tail rows are underpredicted?**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]]
under_rate = (tail["pred"] < tail[target_col]).mean()
under_rate

**Underprediction rate = 99.13% (tail)**

0.9913 means **911 out of 919** tail rows are underpredicted (roughly). So the tail problem is not “high variance”. It is a **systematic downward bias** in the tail regime.

This matches everything we saw earlier:

- tail mean ratio ≈ 0.42
- tail mean bias ≈ −$514
- tail error percentiles mostly negative

**5. How many “catastrophic” misses are driving tail SSE?**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["sq_err"] = (tail["pred"] - tail[target_col])**2

# fraction of tail SSE explained by top k worst rows
for k in [1, 5, 10, 25, 50]:
    share = tail["sq_err"].nlargest(k).sum() / tail["sq_err"].sum()
    print(k, float(share))

**Tail SSE is dominated by a single catastrophic miss**

The SSE concentration is extreme:

- **Top 1 tail row explains 80.85% of tail SSE**
- Top 5 explains 83.64%
- Top 10 explains 84.98%
- Top 50 explains 90.53%

So when we report RMSE in dollars, we are mostly measuring “how bad is the single worst tail miss,” not the typical performance.

That also explains why:

- Dollar R² is low (because SSE is huge from a few points).
- Log-space R² looks strong (because the log compresses the effect of that outlier).

**6. Identify the single worst tail row and inspect it**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["err"] = tail["pred"] - tail[target_col]
tail["abs_err"] = tail["err"].abs()
tail["sq_err"] = tail["err"]**2

worst = tail.sort_values("sq_err", ascending=False).head(1)
worst[["Rndrng_NPI","Year","rbcs_family_desc","Place_Of_Srvc","provider_type","state","ruca_bucket","services",target_col,"pred","err","abs_err"]]

**Worst tail row**

- `rbcs_family_desc` = `Chemotherapeutic Agent`
- `provider_type` = `Radiation Oncology`
- `Place_Of_Srvc` = `O`
- `services` = `53`
- **actual** `stdzd_amt_per_service` = `41,967`
- **predicted** ~`201`
- `err`or `-41,766`

>So the model is behaving like “Chemotherapeutic Agent in this context should cost a few hundred per service,” but the data says “it is forty thousand per service.”

That combination is either:

1. **A real but extremely rare regime** our linear model cannot express from the current feature set (interactions, nonlinearities), or
2. **A coding / mapping / aggregation artifact** (less common, but worth ruling out because the magnitude is so extreme).

Either way, this single point dominating SSE is why our dollar-RMSE looks disastrous while log-space metrics look decent.

Let's check whether it is:

- a weird combination (rare RBCS family + unusual POS + tiny services just above threshold), or
- a data quality oddity (e.g., denominator effect, miscoding), or
- a genuinely extreme but real provider-year outlier.

**7. Quantify “typical tail error” with a robust metric**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["abs_err"] = (tail["pred"] - tail[target_col]).abs()

tail_abs_summary = tail["abs_err"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
tail_abs_summary

**Our tail abs error summary says**

From our tail `abs_err` distribution:

- Median tail miss: **~$309**
- 90th percentile: **~$962**
- 99th percentile: **~$2,370**
- Max: **~$41,766** (the monster)

So for 99 percent of tail rows, our error is in the hundreds to low thousands. Then one row is off by forty thousand and it blows up RMSE and SSE.

**8. Pull the underlying spend totals and other per-service fields for that exact row**

In [ ]:
row_idx = worst.index[0]

eda_df.loc[row_idx, [
    "Rndrng_NPI","Year","rbcs_family_desc","Place_Of_Srvc","provider_type","state","ruca_bucket",
    "services","benes",
    "stdzd_amt_per_service","stdzd_spend",
    "allowed_amt_per_service","allowed_spend",
    "payment_amt_per_service","payment_spend",
    "submitted_charge_per_service","submitted_spend"
]]

**9. Is this NPI consistently extreme, or is 2023 a one-off spike?**

In [ ]:
npi = worst["Rndrng_NPI"].iloc[0]

eda_df.loc[
    (eda_df["Rndrng_NPI"] == npi) & (eda_df["rbcs_family_desc"] == "Chemotherapeutic Agent"),
    ["Year","services","stdzd_amt_per_service","stdzd_spend","Place_Of_Srvc","provider_type","state"]
].sort_values("Year")

In [ ]:
npi

The “catastrophic miss” is a **real, stable, learnable pattern in the data**, not a one-off glitch.

A) It is not a construction artifact

Our per-service and total fields line up:

- `stdzd_spend` ≈ `services * stdzd_amt_per_service`
    
    `53` * `41,967.405094` ≈ `2,224,272.47` (matches our `stdzd_spend`)
    
- `payment_amt_per_service` == `stdzd_amt_per_service` and `payment_spend` == `stdzd_spend`
    
    So the standardized and payment views are consistent.
    
- `allowed_amt_per_service` is even higher (~52.7k), and `submitted_charge_per_service` is higher still (98k).
    
    That “submitted > allowed > paid/standardized” ordering is typical and also internally consistent.
    

So this is a genuine extremely high-cost provider-year-service bucket.

B) It is not a 2023 spike. It is stable across years for this NPI

Same NPI, same service family, same POS, same provider type, same state:

- 2021: ~41,862 per service
- 2022: ~41,705 per service
- 2023: ~41,967 per service

That stability is exactly what we want to see if this is “real behavior” rather than noise.

**10. How extreme is this row relative to its peer group?**

In [ ]:
peer = train_df.loc[
    (train_df["rbcs_family_desc"] == "Chemotherapeutic Agent") &
    (train_df["provider_type"] == "Radiation Oncology") &
    (train_df["Place_Of_Srvc"] == "O"),
    ["stdzd_amt_per_service","services","state","ruca_bucket"]
]

peer["stdzd_amt_per_service"].describe(percentiles=[0.5,0.9,0.95,0.99])

In [ ]:
peer

The peer group is bimodal. Most rows are cheap, a few are ultra-expensive. 

Our peer group is only 18 rows, and the distribution screams “two regimes”:

- median: **~$39.78 per service**
- 90th percentile: **~$12,540**
- 95th percentile: **~$41,729**
- max: **~$41,863**

So in the exact same coarse slice (Chemotherapeutic Agent + Radiation Oncology + POS=O), there are rows clustered around tens of dollars, and a small number clustered around ~42k.

That also explains why our OLS prediction is ~200. With the current feature set, the model mostly learns the dominant “low-cost mode,” and it has no reliable signal to identify which rows belong to the “ultra-expensive mode.”

The next most informative thing to compute is this, using the `test_eval`:
- For that `peer` slice, compare feature values (the numeric covariates) between the ultra-high rows and the low-cost rows. If they are indistinguishable, then we have strong evidence we need a more granular categorical feature (for example `rbcs_cat_subcat`) or a provider-history feature to capture the regime.

1. Let's create a new dataset called `peer_test_eval` from the `test_eval` where we get `rbcs_family_desc` = `"Chemotherapeutic Agent"`, `provider_type` = `"Radiation Oncology"`, `Place_Of_Srvc` = `"O Agent"`:

**1.A. Let's first make a copy of the sliced `test_eval` dataset:**

In [ ]:
peer_test_eval = test_eval.loc[
    (test_eval["rbcs_family_desc"] == "Chemotherapeutic Agent") &
    (test_eval["provider_type"] == "Radiation Oncology") &
    (test_eval["Place_Of_Srvc"] == "O")
].copy()

**1.B. Let's add a a new column `is_worst` that indicates the rows that belong to `npi` from `worst`.**

In [ ]:
peer_test_eval["is_worst"] = peer_test_eval["Rndrng_NPI"].eq(npi)

2. Let's define “ultra-high” vs “low-cost” within the peer slice

This is usually better than using the global top 1% flag, because our peer slice is already narrow.

In [ ]:
num_cols = num_features  # our list

peer = peer_test_eval.copy()

# Define ultra-high and low-cost within this peer slice
hi_cut = peer[target_col].quantile(0.90)   # top 10% within peer
lo_cut = peer[target_col].quantile(0.50)   # bottom 50% within peer

peer["cost_group"] = np.select(
    [peer[target_col] >= hi_cut, peer[target_col] <= lo_cut],
    ["ultra_high", "low_cost"],
    default="middle"
)

peer["cost_group"].value_counts(dropna=False)

3. Let's compare numeric covariates between groups

This produces a compact “are they distinguishable?” table for numeric features:

3.A. Mean/median comparison table

In [ ]:
compare_groups = peer.loc[peer["cost_group"].isin(["ultra_high", "low_cost"])].copy()

summary = (
    compare_groups
    .groupby("cost_group")[num_cols]
    .agg(["mean", "median", "std"])
)

summary

3.B. Add standardized mean difference (best quick signal)

This gives us a single “effect size” number per feature. If SMD is near 0, the groups are basically indistinguishable on that feature.

In [ ]:
def one_vs_group_z(x, group):
    x = float(x)
    g = np.asarray(group, dtype=float)
    mu = np.nanmean(g)
    sd = np.nanstd(g, ddof=1)  # ok because low_cost has n=4 here
    return (x - mu) / sd if sd > 0 else np.nan

hi = peer.loc[peer["cost_group"] == "ultra_high"]
lo = peer.loc[peer["cost_group"] == "low_cost"]

z_tbl = pd.DataFrame({
    "feature": num_cols,
    "ultra_high_value": [hi[c].iloc[0] for c in num_cols],
    "low_cost_mean":    [lo[c].mean() for c in num_cols],
    "low_cost_sd":      [lo[c].std(ddof=1) for c in num_cols],
    "z_vs_low_cost":    [one_vs_group_z(hi[c].iloc[0], lo[c]) for c in num_cols],
}).sort_values("z_vs_low_cost", key=lambda s: s.abs(), ascending=False)

z_tbl

The key interpretation rule:

- **Negative z**: `ultra_high` value is **below** the `low_cost` mean.
- **Positive z**: `ultra_high` value is **above** the `low_cost` mean.
- **Magnitude**:
    - |z| ≈ 0 to 1: not very different
    - |z| ≈ 2: pretty different
    - |z| ≥ 3: extremely different (especially with only 4 `low_cost` rows, this is a strong signal that this point sits far from that group on that feature)

Now our table:

1) `log_services`: z = -7.37 (huge)

- `ultra_high_value` = 3.988984
- `low_cost_mean` = 9.559847
- `low_cost_sd` = 0.756033
- `z_vs_low_cost` = (3.99 - 9.56) / 0.756 ≈ -7.37

Interpretation:

- Within this peer slice, the `ultra_high` row has **much lower `log_services`** than the `low_cost` rows.
- Since `log_services` is log-transformed, this is a massive difference on the original services scale.
- This is a red flag that the `ultra_high` cost-per-service case might be associated with a very different volume regime (even though our `services` column for the worst row was `53`, the `low_cost` rows in this peer slice likely have much higher services if their `log_services` mean is `9.56`, which is extremely large). That suggests we should sanity-check how `log_services` was defined in this dataset.

This single row is telling us: “I’m expensive per service, but I do not have high service volume relative to these `low_cost` rows.”

2) `bene_avg_risk_score`: z = -3.57

- `ultra_high` row’s beneficiaries are **lower risk** than `low_cost` mean by ~3.6 SDs.
- If this holds up, it suggests the extreme cost-per-service is not explained by higher risk score, at least not relative to these low-cost peers.

3) `p_copd`: z = -2.22 and `p_ckd`: z = -1.93

- `ultra_high` row has **lower COPD and CKD prevalence** than `low_cost` peers, relative to the `low_cost` variation.
- Again, this pushes against “this is just sicker patients” as the explanation.

4) `p_cancer6`: z = +1.54

- `ultra_high` has somewhat higher cancer prevalence than `low_cost`, but only ~1.5 SD.
- Not nothing, but not nearly as extreme as the service-volume signal.

5) `log_benes`: z = -1.21

- `ultra_high` row has fewer beneficiaries (or whatever `log_benes` captures) than `low_cost` peers, by ~1.2 SD.

6) `p_htn`, `p_diabetes`, `years_since_enumeration`: z near 0

- These look basically similar between `ultra_high` and `low_cost` within this peer slice.

Within that very narrow peer slice, the ultra-high cost-per-service row is not “high” because the numeric covariates scream “complex population.” Instead it looks like:

- **Lower volume signals (`log_services`, `log_benes`)**
- Some comorbidity rates are lower, not higher
- Cancer prevalence is a bit higher, but not enough to explain a 40k per service situation

That supports the hypothesis we mentioned earlier: **our feature set cannot represent the regime that creates ultra-high per-service costs**, because it is likely driven by something categorical or structural we are not encoding at the right granularity (or by a special pricing/HCPCS subcategory, drug, setting nuance, etc.).

- Just for sanity check, let's look at the `services`, `log_services`, `stdzd_amt_per_service` columns of the `peer` dataframe and calculate services from `log_services` column named `services_from_log`:

In [ ]:
peer.loc[peer["cost_group"] == "low_cost", ["services", "log_services", target_col]] \
    .assign(services_from_log=lambda d: np.expm1(d["log_services"])) \
    .sort_values("services", ascending=False)

Now we know that 

1. The simple OLS gave us R2 of ~0.22 in the original dollar cost scale, but ~0.77 in the log cost scale. This happens because TTR converts the target to log scale before fitting the model. During training the model optimizes the RSS in the log scale. When we make the predictions with `.predict()`, the TTR converts the log predictions to original scale. When we use `.score()`, the TTR predicts in log scale (as explained), converts the values to original scale and calculates the R2.
2. The MAE is ~$32 but RMSE is ~$152 meaning most points are okay but a few are catastropically wrong. 
3. When we group the test dataset by `is_top_1pct_stdzd_amt_per_service` flag (tail is where `is_top_1pct_stdzd_amt_per_service` is `True`), we saw that non-tail had ~104k rows, a MAE of ~$24, RMSE of ~$51, actual mean target of ~$87, and predicted mean target of ~$85; tail had ~919 rows, a MAE of ~$515, RMSE of ~$1532, actual mean target of ~$886, and predicted mean target of ~$372. This meant our model's mean prediction was very close to the actual target mean for the non-tail, but far from the actual target mean for the tail. The maodel could only predict ~42% of the true cost in the tail. 
4. The sum of squared errors in the tail make up ~89% of of the total squared errors for the entire test dataset predictions, meaning the model makes massive errors when predicting the cost in the tail. 
5. When calculate the signed error within the tail, we see that for 99th percentile, the sign of the error is negative, meaning for 99% of the tail, our model is underpredicting. The minimum signed error was ~42k, meaning the model underpredicts some or an observation by ~$42k. This is huge and dominates the RSS, hence the low R2 in the original target scale.
5. When we looked at the top 1, 5, 10, 25 and 50 observations in the tail to see how much of the total tail RSS (i.e., SSE), we saw that the top 1 observation by squared error accounts for ~81% of the entire RSS in the tail. This informed us that there is an observation that explodes RSS.
6. Then we pulled the worst offender by sorting the `test_eval` (similar to `test_df` with extra flags), sorting in descending order, and slicing the top row, we saw that the worst offender has row index of `142102`, `Rndrng_NPI` of `1962539759`, `Year` of `2023` (expected as this is the test set), `rbcs_family_desc` of `Chemotherapeutic Agent`, `Place_Of_Srvc` of `O`, `provider_type` of `Radiation Oncology`, `state` of `IL`, `ruca_bucket` of `Urban`, `services` of `53.0`, `stdzd_amt_per_service` of `41967.405094`, `err` (error) of `-41766.471304`. 
7. We pulled the row index of the worst offender to pull the same index out of the `eda_df` meta dataset for EDA to inspect whether this `stdzd_amt_per_service` of `41967.405094` is a fluke or legit. After inspecting the `allowed_amt_per_service`, `payment_amt_per_service`, and `submitted_charge_per_service`, we realized that the `stdzd_amt_per_service` checks out. 
8. Then we wanted to see if this observation is a "2023-exclusive" observation. When we filted the `eda_df` for the worst offender's `Rndrng_NPI` and `rbcs_family_desc` of `Chemotherapeutic Agent` we realized that the there is a consistent pattern: this provider provided similar service volume in 2021, and 2022 (i.e., `services` of `50` and `62` respectively) with very similar `stdzd_amt_per_service` (i.e., `41862.701000` and `41705.296774`, respectively), meaning this is a legit provider with respectible volume per year. 
9. We then looked at how extreme this observation is relative to its peer group. We defined `peer` by slicing the `train_df` by `rbcs_family_desc` of `"Chemotherapeutic Agent"`, `provider_type` of `"Radiation Oncology"`, `Place_Of_Srvc` of `"O"` and returning the `stdzd_amt_per_service`, `services`, `state`, and `ruca_bucket` columns, we saw that the distribution of `stdzd_amt_per_service` is bimodal. The `peer` dataset had 18 obervations, 2 of which had huge `stdzd_amt_per_service` and the rest were much lower. 
10. Then we created a `peer_test_eval` by creating a copy of the `test_eval` filtered by by `rbcs_family_desc` of `"Chemotherapeutic Agent"`, `provider_type` of `"Radiation Oncology"`, `Place_Of_Srvc` of `"O"`. This way we created `cost_group` variable by labeling each row by either `ultra_high` if the target is within top 10% of the `peer[target_col]`,  `low_cost` if the target is within bottom 50% of the `peer[target_col]` and `middle` for any other value of the target.
11. We then calculated "how far off is the numeric feature of the observation with `cost_group` values of `ultra_high` (n = 1, turns out its the worst offender) from the mean of `cost_group` values of `low_cost`. This told us that the `log_services` value of the `ultra_high` cost is ~7.37 standard deviations lower than the `log_services` value of the `low_cost`; the `log_services` value of the `ultra_high` cost is ~7.37 standard deviations lower than the mean `log_services` value of the `low_cost`; the `bene_avg_risk_score` value of the `ultra_high` cost is ~3.57 standard deviations lower than the mean `bene_avg_risk_score` value of the `low_cost`; the `p_copd` value of the `ultra_high` cost is ~2.22 standard deviations lower than the mean `p_copd` value of the `low_cost`; the `p_ckd` value of the `ultra_high` cost is ~1.93 standard deviations lower than the mean `p_ckd` value of the `low_cost`; the `log_services` value of the `ultra_high` cost is ~7.37 standard deviations lower than the mean `log_services` value of the `low_cost`; the `log_benes` value of the `ultra_high` cost is ~1.21 standard deviations lower than the mean `log_benes` value of the `low_cost`; the `p_cancer6` value of the `ultra_high` cost is ~1.54 standard deviations higher than the mean `p_cancer6` value of the `low_cost`. These indicated that our ultra high cost per service row is not high because the numeric covariates scream complex population. Instead, it looks like lower volume signals (`log_services`, `log_benes`). Some comorbidity rates are lower, not higher. Cancer prevalence is a bit higher but not extreme. 

What should we do next in terms of modeling?

Should we build two models, one for the tail and the other for the non-tail? Should we recreate the train and test sets with more granular level service rbcs family? Should we still move forward with Ridge, XGBoost, and Carboost? Or what else?

Also check if my structured summary above is what you also remember and agree with? 